# Analyse des accidents de la route — Insights clés (BAAC 2024)

## Objectif
Présenter les principaux enseignements issus de l’analyse des accidents corporels de la circulation
en France à partir des données BAAC 2024, dans une logique d’aide à la décision et de prévention.

In [ ]:
import sys
from pathlib import Path

# Ajoute la racine du projet au PYTHONPATH
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
import pandas as pd
from IPython.display import Image, display


DATA = Path("../data/processed")
acc = pd.read_parquet(DATA / "accidents-2024.parquet")
usag = pd.read_parquet(DATA / "usagers-2024.parquet")

print(acc.shape, usag.shape)

In [ ]:
from src.features import add_time_features
acc = add_time_features (acc)

## Insight 1 — Concentration des accidents aux heures de pointe

In [ ]:
from src.utils import save_fig
import matplotlib.pyplot as plt

# Agrégation
acc_per_hour = acc["hour"].value_counts().sort_index()

# Plot
plt.figure(figsize=(8, 4))

plt.plot(acc_per_hour.index, acc_per_hour.values, marker="o", label="Accidents")
plt.plot(
    acc_per_hour.index,
    acc_per_hour.rolling(window=3, center=True).mean(),
    linewidth=3,
    label="Moyenne mobile (3h)"
)

plt.title("Risque d'accident selon l'heure de la journée")
plt.xlabel("Heure")
plt.ylabel("Nombre d'accidents")
plt.xticks(range(0, 24))
plt.grid(axis="y", alpha=0.3)
plt.legend()

plt.show
path = save_fig("01_accidents_par_heure.png")
display(Image(filename=str(path)))

Les accidents sont majoritairement concentrés aux heures de pointe,
notamment le matin et en fin de journée. Cette répartition suggère un lien fort avec les déplacements domicile–travail.


## Insight 2 — Répartition hebdomadaire des accidents

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
acc_per_day = acc["weekday"].value_counts().reindex(order)

plt.figure(figsize=(8, 4))

plt.plot(order, acc_per_day.values, marker="o", linewidth=2)

# Mise en évidence du week-end
plt.axvspan(4.5, 6.5, alpha=0.15, label="Week-end")

plt.title("Variation du nombre d'accidents au cours de la semaine")
plt.xlabel("Jour de la semaine")
plt.ylabel("Nombre d'accidents")
plt.grid(axis="y", alpha=0.3)
plt.legend()

plt.show
path = save_fig("02_accidents_par_jour_semaine.png")
display(Image(filename=str(path)))

Les accidents sont plus fréquents en semaine, avec une baisse notable le dimanche, confirmant l’impact des déplacements professionnels sur la sinistralité routière.

## Insight 3 — Gravité des accidents impliquant les usagers

In [ ]:
order = ["Indemne", "Blessé léger", "Blessé hospitalisé", "Tué"]

counts = (
    usag["grav_label"]
    .value_counts()
    .reindex(order)
    .fillna(0)
    .astype(int)
)

pct = counts / counts.sum() * 100
cum = pct.cumsum()

fig, ax = plt.subplots(figsize=(8, 4))

# Barres horizontales (volumes)
ax.barh(order, counts.values, alpha=0.8)
ax.invert_yaxis()  # "Indemne" en haut

ax.set_title("Gravité des accidents — usagers (volume et structure)")
ax.set_xlabel("Nombre d'usagers")
ax.set_ylabel("Gravité")
ax.grid(axis="x", alpha=0.3)

# Annotations en %
for i, (v, p) in enumerate(zip(counts.values, pct.values)):
    ax.text(v, i, f"  {p:.1f}%", va="center")

# Courbe cumulée sur axe secondaire (option premium)
ax2 = ax.twiny()
ax2.plot(cum.values, range(len(order)), marker="o", linewidth=2)
ax2.set_xlim(0, 100)
ax2.set_xlabel("Part cumulée (%)")

plt.show
path = save_fig("03_gravite_usagers.png")
display(Image(filename=str(path)))

La majorité des usagers impliqués dans les accidents sont indemnes ou légèrement blessés.
Les accidents mortels restent minoritaires, mais représentent un enjeu majeur en termes de prévention.

## Insight 4 — Périodes horaires à risque élevé

In [ ]:
from src.features import merge_usagers_accidents

usag_acc = merge_usagers_accidents(usag, acc)

# Filtrage accidents mortels
fatal_per_hour = (
    usag_acc.loc[usag_acc["grav_label"] == "Tué", "hour"]
    .value_counts()
    .sort_index()
)

# Courbe brute
plt.figure(figsize=(8, 4))
plt.plot(
    fatal_per_hour.index,
    fatal_per_hour.values,
    marker="o",
    label="Décès par heure"
)

# Moyenne mobile (fenêtres rares → indispensable)
plt.plot(
    fatal_per_hour.index,
    fatal_per_hour.rolling(window=3, center=True).mean(),
    linewidth=3,
    label="Moyenne mobile (3h)"
)

plt.title("Risque de mortalité selon l'heure de la journée")
plt.xlabel("Heure")
plt.ylabel("Nombre de personnes tuées")
plt.xticks(range(0, 24))
plt.grid(axis="y", alpha=0.3)
plt.legend()

plt.show
path = save_fig("04_accidents_mortels_par_heure.png")
display(Image(filename=str(path)))

Le croisement entre l’heure de l’accident et la gravité met en évidence des périodes
à risque élevé, durant lesquelles les accidents mortels sont plus fréquents.

La similarité entre la distribution horaire des accidents et celle des décès indique que la mortalité est largement corrélée à l’exposition. En revanche, l’analyse des ratios met en évidence des créneaux horaires où la probabilité de décès par accident est plus élevée

Ces créneaux pourraient constituer des cibles prioritaires pour des actions de prévention renforcées.

## Insight 5 — Top Départements

In [ ]:
import geopandas as gpd
from src.features import accidents_par_departement, merge_accidents_geo

geo_path = PROJECT_ROOT / "data" / "geo" / "departements.geojson"
geo_dep = gpd.read_file(geo_path)
acc_dep = accidents_par_departement(acc)
geo_dep_acc = merge_accidents_geo(geo_dep, acc_dep)

plt.figure(figsize=(8, 10))

geo_dep_acc.plot(
    column="nb_accidents",
    cmap="Reds",
    linewidth=0.3,
    edgecolor="black",
    legend=True,
    legend_kwds={"label": "Nombre d'accidents"},
)

plt.title("Accidents de la route par département – France (2024)")
plt.axis("off")

path = save_fig("05_carte_accidents_par_departement.png")
display(Image(filename=str(path)))

Les départements les plus peuplés concentrent logiquement le plus grand nombre d’accidents. Cette analyse en volume brut met en évidence des zones prioritaires en termes d’exposition, mais nécessite une normalisation par la population ou le trafic pour évaluer la dangerosité relative.